# Needle FP4 · Training sensitivity and packed inference
Shreya Verma · Anshuman Mishra · Nidhi Prakash | Group 59, slot B6 | CMU 10-414/714

**Run this report in a fresh Google Colab runtime.** The setup cell selects a repository revision. Add a read-only `GITHUB_TOKEN` Colab secret for private GitHub access. A GPU runtime enables the CUDA tests and event timings. Download the results ZIP before disconnecting. All empirical tables, plots and measurements in this notebook are generated during execution.

**Questions.** Which of a linear layer's three matrix products changes learning most when its operands use NVFP4 precision? How do real four-bit weights affect storage, prediction accuracy and batch-one latency?


## 1 · Why change precision inside Needle?
A classifier maps image pixels to class scores. The loss compares those scores with the correct label. A gradient tells the optimizer how a small change to each weight would change the loss. Training updates the weights repeatedly. Inference uses the trained weights to predict a label. These stages place different demands on a numerical representation. A small error in a prediction is visible immediately. An error in a training signal can affect every later update.

Needle provides tensors, differentiation, layers, optimizers and native array backends. We extend that existing stack in two places. A custom linear operation lets us select the numerical inputs to each matrix product in training. A packed matrix type gives inference its own byte layout and kernels. This lets us investigate accuracy and memory traffic separately. Our hypotheses are that the location of rounding matters for training and that smaller weight buffers create an opportunity for faster batch-one inference. The measurements below determine whether those hypotheses hold for this model and device.


## 2 · What four-bit floating point represents
A four-bit code can take sixteen bit patterns. E2M1 uses one sign bit, two exponent bits and one fraction bit. Its positive magnitudes, including zero, are 0, 0.5, 1, 1.5, 2, 3, 4 and 6. Positive and negative zero have the same numerical value. Real weights rarely land exactly on this grid. Quantization selects a nearby code and loses some numerical information. Packing two selected codes into one byte preserves those codes exactly.

NVFP4 extends the range by giving each block of sixteen values an eight-bit E4M3 scale. There is also a float32 scale for the tensor. If A is the largest absolute value in the tensor, the tensor scale is A / (6 × 448). If B is the largest magnitude in a block, the block scale is (B / 6) divided by the tensor scale and rounded to E4M3. Reconstruct a weight by multiplying its signed grid value by both scales. The implementation handles a zero tensor, a block whose rounded scale is zero, and a final block shorter than sixteen. The extra range comes with a measurable storage cost.

Nearest rounding makes the same decision every time. For a normalized scalar of 0.2, it always chooses zero. Stochastic rounding chooses 0.5 with probability 0.4 and zero otherwise. Its expected value is 0.2. This reasoning applies to an in-range scalar with fixed neighbors. Scale rounding, clipping, matrix products and activation functions still influence the model's gradients. Comparing trained models tests whether changing rounding makes a practical difference.

NVFP4 training research motivates the block format and stochastic gradient rounding. MXFP4 provides an alternative with a power-of-two scale for each block of 32. Our comparison asks what these individual choices do in a small classifier. The optional weight modes adapt ideas about forward/backward consistency. They implement individual mechanisms and use different precision choices from some full published recipes. See the references at the end.


## 3 · How a linear layer connects to the backend
For a batch X with shape (B, I) and weights W with shape (I, O), Y = XW has shape (B, O). Each output sums I terms. During backward, dY is the incoming derivative of loss with respect to Y. Multiplying dY by W transposed produces dX, the derivative that earlier layers need. Multiplying X transposed by dY produces dW, the derivative the optimizer needs. The first sums across O outputs and the second across B examples. These different summation axes determine which values share a block scale.

| Component | Entry point | Design decision |
|---|---|---|
| Reference | `python/needle/fp4/reference.py` | Define rounding, scale encoding, byte order and a reproducible GEMV reduction in a readable implementation. |
| Numerical simulation | `python/needle/fp4/quantize.py` | Round and reconstruct with Needle arrays. New backend floor and absolute-value operations support rounding. |
| Autodiff and layer | `python/needle/ops/ops_fp4.py`, `nn/nn_fp4.py`, `fp4/config.py` under `python/needle` | Register a custom backward rule and preserve the existing Linear parameter and bias interface. |
| Packed storage | `python/needle/fp4/packed.py`, `src/nvfp4.h`, native CPU/CUDA backends | Store two codes per byte and one scale byte per block. Decode them during matrix–vector multiplication. |

Training uses fake quantization. Each selected operand is rounded and reconstructed, then an ordinary float32 product combines the values. The trainable parameters stay in float32. Because literal rounding has a zero derivative almost everywhere, the custom operator uses a straight-through rule. It starts its backward calculation from the original X and W, then applies the configured rounding separately to Dgrad and Wgrad. The base SR switch affects the incoming gradient dY. A separate optional mode re-quantizes the forward weights stochastically for Dgrad.

Inference uses actual byte buffers. Needle stores a linear weight as (input, output), so the inference wrapper transposes it once to create contiguous rows of (output, input). The CUDA GEMV assigns four output rows to a warp, reuses a block of input features, decodes the nibble values and scales, and combines partial sums in a specified order. A specialized dense kernel provides a comparable baseline. The NumPy reference shares the reduction order so the tests can distinguish format errors from ordinary floating-point summation differences.


### Choose the run before starting
The default demo pairs seed 0 across three epochs, using 20,000 training examples on CPU or 50,000 on GPU. It shows placement effects and supplies the FP32 checkpoint for the format comparison. Set FULL_STUDY to True in a fresh runtime for the registered experiment as well. That adds the eight main configurations, four sensitivity settings and gradient diagnostics on all of CIFAR-10. Budget for a long run. Results and command logs are written under the printed run directory. Download the ZIP before disconnecting. A terminated runtime can lose local files.


In [ ]:
from pathlib import Path
import base64, csv, hashlib, json, math, os, shutil, subprocess, sys, tarfile, tempfile, urllib.request, uuid, zipfile
from datetime import datetime, timezone
from IPython.display import display, Markdown, Image, FileLink

# AI-assisted report code: ChatGPT/Codex (GPT-6), adapting the team's Claude-assisted notebook.
FULL_STUDY = False
CODE_REF = 'c11a393db74e1752df7d4a5268e1ba2ae3b447ab'  # immutable implementation under evaluation
PRIVATE_CODE_URL = ""  # private Drive download shared with graders, or a private repo archive URL
SEEDS = [0]
EPOCHS = 3
EXTRA_DIAGNOSTICS = FULL_STUDY
START = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
RUN = START / ('needle-results-' + uuid.uuid4().hex[:10])
RUN.mkdir()
checkout = Path(tempfile.mkdtemp(prefix='needle-source-')) / 'fp4-needle'
url = 'https://github.com/shreya-v7/fp4-needle.git'
token = os.environ.get('GITHUB_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except (ImportError, KeyError):
        pass
    except Exception as exc:
        print('Colab secret unavailable:', type(exc).__name__)
git_env = os.environ.copy()
if token:
    basic = base64.b64encode(('x-access-token:' + token).encode()).decode()
    git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraheader', GIT_CONFIG_VALUE_0='Authorization: Basic ' + basic)
git_env['GIT_TERMINAL_PROMPT'] = '0'
subprocess.run(['git','clone','--quiet','--no-checkout',url,str(checkout)],env=git_env,check=True)
del git_env, token
if 'basic' in globals(): del basic
subprocess.run(['git','-C',str(checkout),'checkout','--quiet','--detach',CODE_REF],check=True)
def sha(ref):
    return subprocess.check_output(['git','-C',str(checkout),'rev-parse',ref],text=True).strip()
REVISION = sha('HEAD')
assert REVISION == sha(CODE_REF + '^{commit}')
print('Running commit:', REVISION, '| main at clone:', sha('origin/main'))
os.chdir(checkout)
def run(args, log=None):
    print('Running', *args, flush=True)
    with (RUN / 'commands.jsonl').open('a') as audit:
        audit.write(json.dumps([str(x) for x in args]) + '\n')
    if log is None:
        subprocess.run([str(x) for x in args],check=True)
        return
    with (RUN / log).open('w') as stream:
        proc = subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in proc.stdout:
            print(line,end=''); stream.write(line)
        if proc.wait(): raise RuntimeError(f'FAILED: {log}; inspect {RUN / log} for partial evidence')
run([sys.executable,'-m','pip','install','-q','pybind11','cmake>=3.27,<4','numpy','pytest','matplotlib'])
cmake = shutil.which('cmake')
run([cmake,'-S','.','-B','build','-DPython_EXECUTABLE='+sys.executable])
run([cmake,'--build','build','--parallel','2'])
os.environ['PYTHONPATH'] = str(checkout/'python')
os.environ['NEEDLE_BACKEND'] = 'nd'
sys.path.insert(0,str(checkout/'python'))
import needle as ndl
import matplotlib.pyplot as plt
DEVICE = 'cuda' if ndl.cuda().enabled() else 'cpu'
gpu_info = ndl.cuda().device_info() if DEVICE == 'cuda' else None
SUBSET = 0 if DEVICE == 'cuda' else 20000
meta = dict(revision=REVISION, main_at_clone=sha('origin/main'), device=DEVICE, gpu=gpu_info, seeds=SEEDS,
            full_study=FULL_STUDY,
            registered=dict(dataset='CIFAR-10', train_examples=50000, test_examples=10000,
                            main_seeds=list(range(5)), epochs=20, lr_seeds=[0,1], depth_seeds=[0,1,2],
                            sweep_epochs=10, grad_checkpoints=[0,391,1173], grad_seed=0, sr_draws=32),
            gemv_seed='rows * 100003 + cols', mnist=dict(seed=0, train_examples=60000, test_examples=10000, epochs=3),
            cifar_test_examples=10000, epochs=EPOCHS, train_examples=SUBSET or 50000, diagnostics=EXTRA_DIAGNOSTICS,
            utc=datetime.now(timezone.utc).isoformat())
(RUN/'run.json').write_text(json.dumps(meta,indent=2))
print(json.dumps(meta,indent=2))

print('Run directory:', RUN)
if DEVICE == 'cpu': print('CUDA tests and GPU timings skipped. Benchmark device: CPU.')


## Evaluation · Does the implementation agree with its numerical reference?
The tests cover the code grid, ties, signed values, packing order, block scales, odd sizes, rounding, gradients and kernel outputs. A CPU run can only establish the checks available on that runtime. Pytest's exit code controls whether this cell succeeds.


In [ ]:
run([sys.executable,'-m','pytest','tests/fp4','-q','-rs'],'tests.txt')


### Set up the image data
A CIFAR-10 image becomes 3,072 input numbers. The classifier maps them through two hidden layers to ten class scores. MNIST is a separate handwritten-digit classification task for inference. File hashes below identify datasets. They are inputs to the experiment.


In [ ]:
data = checkout/'data'
data.mkdir(exist_ok=True)
def fetch(url, name, digest):
    path = data/name
    if not path.exists(): urllib.request.urlretrieve(url,path)
    with path.open('rb') as f:
        actual=hashlib.file_digest(f,'md5').hexdigest()
    if actual != digest: raise ValueError(f'Unexpected data file: {name}')
    return path
cifar = fetch('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz','cifar-10-python.tar.gz','c58f30108f718f92721af3b95e74349a')
with tarfile.open(cifar) as archive: archive.extractall(data,filter='data')
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}

for name, digest in MNIST.items():
    fetch('https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/'+name,name,digest)
print('Training and test data ready.')


### Which multiplication changes learning?
For an input batch X and a weight matrix W, the forward pass computes XW. Backward uses the incoming error signal to calculate an input gradient and a weight gradient. Switch one placement at a time, keep the architecture, seed and batch order paired, and compare final test accuracy. Here SR randomizes the rounding of the incoming gradient. Model updates and matrix accumulation use float32.


In [ ]:
CONFIGS = ['fp32','fprop','dgrad','dgrad+sr','wgrad','wgrad+sr','fprop+dgrad+wgrad','fprop+dgrad+wgrad+sr']
final = {}
for seed in SEEDS:
    path = RUN/f'cifar_s{seed}.csv'
    args = [sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--configs','all','--only',*CONFIGS,
            '--device',DEVICE,'--epochs',EPOCHS,'--seed',seed,'--train-subset',SUBSET,'--out',path,
            '--save-weights',str(RUN/'ckpt'/'cifar_{config}_s{seed}.npz')]
    run(args, f'training-s{seed}.txt')
    with path.open() as f:
        observations = [r for r in csv.DictReader(f) if r['split'] == 'test']
    for name in CONFIGS:
        epochs = [int(r['epoch']) for r in observations if r['config'] == name and int(r['seed']) == seed]
        print(name, 'seed', seed, 'completed epochs', epochs)
        if epochs != list(range(1, EPOCHS + 1)):
            raise RuntimeError(f'Partial demo for {name}; see {path}. No comparison available.')
    rows = [r for r in observations if int(r['epoch']) == EPOCHS]
    by_name = {r['config']:100*float(r['accuracy']) for r in rows}
    if len(rows)!=len(CONFIGS) or set(by_name)!=set(CONFIGS) or any(not math.isfinite(v) for v in by_name.values()):
        raise RuntimeError(f'Incomplete or invalid final epoch for seed {seed}; inspect {path}')
    final[seed]=by_name


In [ ]:
from statistics import mean, pstdev
summary=[]
for name in CONFIGS:
    values=[final[s][name] for s in SEEDS]
    delta=mean(final[s][name]-final[s]['fp32'] for s in SEEDS)
    summary.append((name,mean(values),pstdev(values) if len(values)>1 else None,delta))
print('Device', DEVICE, '| seed(s)', SEEDS, '| examples', SUBSET or 50000, '| final epoch', EPOCHS)
print('configuration              accuracy (%)    seed SD (pp)     paired gap vs FP32 (pp)')
for name,avg,sd,delta in summary:
    sd_text=f'{sd:.2f}' if sd is not None else 'single seed'
    print(f'{name:27s} {avg:9.2f} {sd_text:>14s} {delta:+18.2f}')
for name in ['dgrad','wgrad','fprop+dgrad+wgrad']:
    shift=mean(final[s][name+'+sr']-final[s][name] for s in SEEDS)
    print(f'{name} · SR minus nearest: {shift:+.2f} percentage points')
fig,ax=plt.subplots(figsize=(9,4))
ax.barh([x[0] for x in summary],[x[3] for x in summary],color='#257b78')
ax.axvline(0,color='black',lw=1); ax.set_xlabel('Final test accuracy minus paired FP32 baseline (percentage points)')
ax.invert_yaxis(); fig.tight_layout(); fig.savefig(RUN/'training.png'); plt.show()


### Does packing change batch-one inference?
A matrix-vector product (GEMV) is one layer applied to one input vector. Compare the two dedicated kernels on the same shape and device. The speed ratio is dense time divided by packed time. Above one favors packed execution. Below one favors dense execution. Buffer sizes include scales and padding. The bandwidth estimate uses minimum bytes moved. Training and initial packing are outside this kernel timer.


In [ ]:
sizes = [1024,2048,4096] + ([8192] if DEVICE=='cuda' else [])
run([sys.executable,'benchmarks/bench_gemv.py','--device',DEVICE,'--sizes',*sizes,*(['--shapes','11008x4096','4096x11008'] if DEVICE=='cuda' else []),'--out',RUN/'gemv.csv'],'gemv.txt')
with (RUN/'gemv.csv').open() as f: rows = {r['kernel']:r for r in csv.DictReader(f) if r['shape']=='4096x4096'}
a,b = rows['fp32 gemv (ours)'],rows['nvfp4 gemv (ours)']
print(f"Device {DEVICE} · 4096 × 4096 · dense {float(a['time_us']):.2f} µs · packed {float(b['time_us']):.2f} µs")
print(f"Dense/packed time: {float(a['time_us'])/float(b['time_us']):.3f}× · dense/packed weight bytes: {float(a['weight_MiB'])/float(b['weight_MiB']):.3f}×")
display(Image(filename=str(RUN/'gemv.png')))


### What does weight approximation do to predictions?
Train an FP32 classifier on MNIST and pack the same weights. The test-set accuracy path expands the quantized weights for batched matrix multiplication. The separate single-image path uses packed GEMVs for latency and prediction agreement. Weight bytes refer to the representation. Total process memory includes other buffers.


In [ ]:
run([sys.executable,'apps/fp4_inference.py','--device',DEVICE],'inference.txt')


### Compare formats on the identical checkpoint
The next table evaluates NVFP4 and MXFP4 weight reconstruction and test accuracy. SQNR is a log ratio of original weight energy to error energy. Larger SQNR means closer reconstruction. This comparison uses simulated quantized weights.


In [ ]:
ckpt = RUN/'ckpt'/f'cifar_fp32_s{SEEDS[0]}.npz'
run([sys.executable,'apps/fp4_mxfp4.py',ckpt,'--device',DEVICE,'--out',RUN/'formats.md'])
display(Markdown((RUN/'formats.md').read_text()))


### Optional mechanism checks
The gradient diagnostic freezes weights and an evaluation batch, measures differences from FP32 gradients, and averages stochastic draws. The weight experiment varies whether forward and backward share tile scales or re-quantize the forward weights. Enable the flag at the top to collect fresh evidence on these questions.


In [ ]:
if EXTRA_DIAGNOSTICS:
    run([sys.executable,'apps/fp4_grad_error.py','--dataset','cifar10','--device',DEVICE,
         '--checkpoints',0,391,1173,'--out',RUN/'gradients.csv'],'gradients.txt')
    for seed in SEEDS:
        run([sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--device',DEVICE,'--configs','weights',
             '--epochs',EPOCHS,'--seed',seed,'--train-subset',SUBSET,'--out',RUN/f'weights_s{seed}.csv'],f'weights_s{seed}.txt')
else:
    print('Optional gradient and weight experiments skipped in this run.')


In [ ]:
if EXTRA_DIAGNOSTICS:
    print('Gradient relative errors, first hidden layer; mean column uses the configured draw count')
    with (RUN/'gradients.csv').open() as f:
        for r in csv.DictReader(f):
            if r['layer']=='layer1' and r['config'] in ('dgrad','dgrad+sr'):
                print(r['step'],r['config'],'one draw',r['rel_error'],'mean',r['rel_error_of_mean'],'draws',r['sr_draws'])
    print('Weight-mode experiment, final-epoch accuracy (%)')
    for seed in SEEDS:
        with (RUN/f'weights_s{seed}.csv').open() as f:
            rows=[r for r in csv.DictReader(f) if r['split']=='test' and int(r['epoch'])==EPOCHS]
        if len({r['config'] for r in rows})!=4: raise RuntimeError(f'Incomplete weight-mode run for seed {seed}')
        for r in rows: print(seed,r['config'],round(100*float(r['accuracy']),2))


### Preregistered CIFAR-10 experiment
Metric A averages held-out accuracy in the last three scheduled epochs. Subtract metric A within each seed before computing its mean, sample SD and paired standard error. The main study fixes width 256, two hidden layers, batch 128, SGD momentum 0.9, learning rate 0.05, all 50,000 training images and seeds 0 through 4. Each configuration gets 20 epochs. A separate random stream supplies SR. The two Dgrad operand controls keep either W or dY in FP32. Their comparison distinguishes a changed learning signal from the effect of regrouping the weights.

H1 requires SR to beat nearest by more than two paired standard errors, with at least four positive seeds for each of Dgrad and all-three. H1b requires both SR means to be within one percentage point of FP32. H2 requires rounding dY alone to lose more than one point and two paired standard errors, while W-only and dY-only with SR each remain within one point of FP32. H3 compares the mean FP32-minus-Dgrad gap at epochs 3 and 20. An equal or larger final gap counts as genuine failure, a final gap below half the early gap as slower learning, and intermediate values as partial closure.

The sensitivity runs change only learning rate (0.01 or 0.1, seeds 0 and 1) or depth (one or four hidden layers, seeds 0 through 2). They use ten epochs, scoring epochs 8–10. The matching default setting uses the first ten epochs of the main study. The directional criterion is a positive SR gain exceeding two paired standard errors. An observed non-finite training loss counts as chance from that epoch forward. Interrupted jobs and missing measurements receive a partial status. The original registered rules are in `docs/PREREGISTRATION.md`.


In [ ]:
from collections import defaultdict
import shlex
sys.path.insert(0, str(checkout / 'apps'))
from fp4_stats import main_report, sweep_report, plot_curves

def checked_series(path, configs, seed, horizon):
    scores = defaultdict(list)
    diverged = {}
    if path.exists():
        with path.open() as stream:
            for row in csv.DictReader(stream):
                name, epoch = row['config'], int(row['epoch'])
                if name not in configs or int(row['seed']) != seed:
                    raise ValueError(f'Unexpected identity in {path}')
                if row['split'] == 'train' and not math.isfinite(float(row['loss'])):
                    diverged[name] = epoch
                if row['split'] == 'test':
                    accuracy = float(row['accuracy'])
                    if not 0 <= accuracy <= 1 or not math.isfinite(float(row['loss'])):
                        raise ValueError(f'Invalid test row in {path}')
                    scores[name].append((epoch, 100 * accuracy))
    result = {}
    for name in configs:
        rows = scores[name]
        end = diverged.get(name, horizon + 1)
        observed = [e for e, _ in rows]
        print(path.stem, name, 'measured epochs', observed, '/', horizon)
        if not 1 <= end <= horizon + 1 or observed != list(range(1, end)):
            raise ValueError(f'Partial or duplicate epochs in {path.name}: {name}, seed {seed}')
        values = [a for _, a in rows]
        if name in diverged:
            print(f'{name} diverged in epoch {end}; assigning chance through epoch {horizon}.')
            values += [10.0] * (horizon - end + 1)
        result[name, seed] = values
    return result

if FULL_STUDY:
    study = RUN / 'extended'
    study.mkdir(exist_ok=False)
    groups = defaultdict(dict)
    # The versioned job list sets the configurations, seeds, depth, lr and epoch count.
    lines = (checkout / 'scripts/extended_jobs.txt').read_text().splitlines()
    (study / 'planned-jobs.txt').write_text('\n'.join(lines))
    for line in lines:
        args = shlex.split(line)
        output = study / Path(args[args.index('--out') + 1]).name
        args[args.index('--out') + 1] = str(output)
        seed = int(args[args.index('--seed') + 1])
        horizon = int(args[args.index('--epochs') + 1])
        configs = (args[args.index('--only')+1:args.index('--epochs')] if '--only' in args
                   else ['fp32','dgrad','dgrad+sr'])
        command = [sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--device',DEVICE,*args]
        print('CIFAR-10, 50,000 training images;', REVISION, DEVICE, flush=True)
        run(command, f'extended/{output.stem}.log')
        groups[output.stem.split('_seed')[0]].update(checked_series(output, configs, seed, horizon))
    main = groups['main']
    text = main_report(main)
    for prefix, default, label in [('lr','0.05','Learning rate'),('depth','2','Hidden depth')]:
        sweep = {k[len(prefix):]: v for k,v in groups.items() if k.startswith(prefix)}
        text += sweep_report(label, sweep, main, default)
    report = '\n'.join(text)
    (study / 'verdicts.md').write_text(report)
    display(Markdown(report))
    plot_curves(main, study / 'curves.png')
    display(Image(filename=str(study / 'curves.png')))
else:
    print('Skipped full study: zero registered jobs executed; H1, H1b, H2, H3 and sensitivity unscored.')


## Discussion · What the output supports
Read the isolated Fprop, Dgrad and Wgrad rows first. Compare each to FP32 at the same epoch on the same seed. The all-three condition combines effects and cannot by itself identify the sensitive operation. Then compare SR with nearest for the same placement. Report the change in percentage points and the remaining gap from baseline. Across several seeds, inspect the direction and spread of paired differences. A single seed is a demonstration at one initialization.

For the optional diagnostic, weights and an evaluation batch are fixed while gradient rounding changes. A reduced error after averaging stochastic draws is evidence that some error varies across draws. A finite average still has sampling variability, and other rounded operands can introduce persistent error. The diagnostic measures weight gradients at individual layers. An input-gradient approximation can therefore appear in the measured weight gradient of an earlier layer.

The packed path measures weight representation bytes, kernel time and model-level single-image latency as separate quantities. The process can retain both models at once. Cache reuse also changes actual traffic relative to an estimate that counts W, x and y once. More compact weights can lose time to decoding. Report the observed ratio in either direction and identify the hardware, shape and revision. A matrix–vector kernel in an image classifier isolates a building block also used in decoding. It does not measure language-model tokens per second.

The experiments cover small multilayer perceptrons and horizons of up to twenty epochs. Longer training, architectures with different activation statistics and native FP4 tensor-core training could behave differently. A useful follow-up would compare precision policies on those workloads using the same matched-baseline design.


## References


1. NVIDIA. [Pretraining Large Language Models with NVFP4](https://arxiv.org/abs/2509.25149). 2025.
2. Chmiel, Fishman, Banner, Soudry. [FP4 All the Way: Fully Quantized Training of LLMs](https://arxiv.org/abs/2505.19115). 2025.
3. Tseng, Yu, Park. [Training LLMs with MXFP4](https://arxiv.org/abs/2502.20586). 2025.
4. Castro et al. [Quartet: Native FP4 Training Can Be Optimal for Large Language Models](https://arxiv.org/abs/2505.14669). 2025.
5. DeepSeek-AI. [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437). 2024.
6. Open Compute Project. OCP Microscaling Formats (MX) Specification v1.0. 2023.
7. Datasets: MNIST (LeCun et al., 1998); CIFAR-10 (Krizhevsky, 2009).
8. Cook and Thinking Machines Lab. [End-to-End 4-Bit Training](https://thinkingmachines.ai/blog/end-to-end-4-bit-training/). 2026.


## AI assistance, verification and code access
The team used Anthropic Claude Code (the project session displays Opus 5.5) for the FP4 reference, quantizer, layer, autograd operator, native kernel sections, experiment runners, tests, CI, related-work search and drafting. Markers in the source identify assisted code, including the FP4 changes under `python/needle`, `src`, `apps`, `scripts`, `benchmarks` and `tests/fp4`. The baseline Needle implementation comes from course homework. OpenAI ChatGPT/Codex (GPT-6 for this revision) assisted with the report narrative, execution cells, provenance checks and README instructions. The report revision received structural and syntax checks. The test log, commands and generated tables below document verification performed by the person executing this notebook. Each author should review the run and confirm their own contributions before submitting.

The setup checks out the pinned implementation revision. The next cell creates an archive from that revision. Put its private staff-accessible URL in PRIVATE_CODE_URL before submitting. For submission, provide staff authenticated private access to the download and to the executed notebook. Confirm both from a staff-accessible account.


In [ ]:
# Export only tracked source and tests at the exact evaluated revision.
source_zip = START / f'fp4-needle-{REVISION[:12]}-source.zip'
subprocess.run(['git','archive','--format=zip',f'--output={source_zip}',REVISION],check=True)
if PRIVATE_CODE_URL:
    display(Markdown(f'[Private code download for staff]({PRIVATE_CODE_URL})'))
else:
    print('Code sharing pending: privately share the source ZIP with staff and add its URL here before submission.')
print('Source archive:', source_zip)


## Export this run
The archive contains the settings, test log, measurements and figures. Checkpoint arrays stay in the run directory. Read the final test rows for each configuration, identify the device on the timing, and describe the comparison even when the outcome is mixed.


In [ ]:
archive = START/(RUN.name+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and path.suffix != '.npz': z.write(path,path.relative_to(RUN))
print('Results:',archive)
os.chdir(START)
display(FileLink(archive.name))
